# Adaptive Compute SSM — Phase 1: Recurrent Monolithic Processing Layer
## Architecture Design and Training on TinyStories

---

## Abstract

This notebook implements Phase 1 of the **Adaptive Compute SSM-MoE Architecture**: a language model
that combines a State Space Model (SSM) backbone with a single shared processing layer applied
*recurrently*. The key insight is that not all tokens require the same amount of computation.
The model learns to use fewer processing steps when the answer is obvious and more when it is not.

In this first phase we use a **monolithic MLP** as the compute layer (no Mixture-of-Experts yet).
This isolates and validates three core mechanisms before adding routing complexity:

1. **SSM state as the only inter-token memory** — the recurrent processing stack works on a
   stateless scratch space; the SSM state is read once (input layer) and written once (output layer)
   per token.
2. **Shared processing layer weights** — depth comes from repeated application of the *same*
   parameters, not from distinct layer stacks. With K active experts per pass and D recurrent passes
   the effective computation path space grows combinatorially (Phase 2+).
3. **Differentiable depth gating via KS anchoring** — the gate MLP produces scores trained to
   follow N(0,1); routing probability is Φ(s), the standard-normal CDF. The full-unroll training
   strategy weights every exit depth's loss by its probability mass, requiring no REINFORCE
   or straight-through estimator.

### Phase roadmap
| Phase | What we add |
|-------|-------------|
| **1 (this notebook)** | SSM + monolithic MLP + depth gate + KS anchoring + log-norm regularization |
| 2 | Retrieval-based MoE router + heterogeneous expert types |
| 3 | Expert birth/death lifecycle (marginal utility index) |
| 4 | Inference-time Smirnov depth control (μ shift) |
| 5 | Scale to full pre-training corpus |

In [ ]:
# Install all dependencies first
# comet_ml must be installed and imported BEFORE PyTorch
!pip install -q comet_ml datasets transformers tqdm huggingface_hub

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  IMPORTANT: comet_ml MUST be imported before PyTorch for full logging   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from comet_ml import start
from comet_ml.integration.pytorch import log_model

import math
import os
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import login
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from google.colab import userdata

# HuggingFace login (needed for dataset access)
HF_KEY = userdata.get('HF_KEY')
login(token=HF_KEY)
COMET_KEY = userdata.get('COMET_KEY')
# Comet ML experiment
experiment = start(
    api_key=COMET_KEY,
    project_name='general',
    workspace='irsotarriva'
)
experiment.set_name('AdaptiveSSM-Phase1-MonolithicMLP')
print('Experiment started:', experiment.get_key())

---
## Architecture Overview

The model consists of **three strictly separated layer types** that are never interchangeable:

```
Token t
  │
  ▼
┌─────────────────────────────────────────────────────────────┐
│  INPUT LAYER  (once per token)                              │
│  z₀ = LayerNorm( W·[e_t ; h_{t-1}] )                       │
│  Reads SSM state h_{t-1}. Never accessed again.             │
└──────────────────────────┬──────────────────────────────────┘
                           │ z₀
                           ▼
              ┌────────────────────────┐
              │  GATE MLP              │  s₀ = gate(z₀)
              │  P(deeper) = Φ(s₀)    │──► continue? ──┐
              └────────────────────────┘                │
                           │ exit                       │ continue
                           │                            ▼
                           │             ┌──────────────────────────┐
                           │             │  PROCESSING LAYER        │
                           │             │  z₁ = z₀ + MLP(LN(z₀))  │
                           │             │  (shared weights)        │
                           │             └──────────┬───────────────┘
                           │                        │ z₁
                           │                    gate(z₁) → ...
                           │
                    z_K (final depth)
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│  OUTPUT LAYER  (once per token)                             │
│  logits = W_lm · LN(z_K)                                   │
│  h_t = gate·h_{t-1} + (1-gate)·tanh(W_c·z_K)              │
│  Writes SSM state h_t. Never accessed until next token.     │
└─────────────────────────────────────────────────────────────┘
```

### Key invariant
The SSM state `h` is **read exactly once** (input layer) and **written exactly once**
(output layer) per token. The processing layers operate in a **stateless scratch space**
and have zero access to `h`. This separation means the processing layer is fully agnostic
to sequence modeling dynamics and can be reused across depths without ambiguity.

---
## 1. Input Layer

The input layer is the **sole interface between the SSM memory and the processing stack**.
It receives two signals:

- **e_t** — the embedding of the current token (vocab lookup, shape `[batch, d_embed]`)
- **h_{t-1}** — the SSM state carrying compressed sequence history (shape `[batch, d_state]`)

These are concatenated and projected into the *internal processing space* `d_internal`:

```
z₀ = LayerNorm( W_in · [e_t ; h_{t-1}] + b_in )
```

The LayerNorm stabilizes the scale of `z₀` before it enters the depth loop, which is
important because `h_{t-1}` and `e_t` may live at very different scales early in training.

This layer is **executed exactly once per token**, regardless of how many processing steps follow.

In [ ]:
class InputLayer(nn.Module):
    """
    Reads SSM state h_{t-1} and token embedding e_t.
    Projects them into the internal processing space z_0.
    Executed exactly once per token.
    """
    def __init__(self, d_embed: int, d_state: int, d_internal: int):
        super().__init__()
        self.proj = nn.Linear(d_embed + d_state, d_internal, bias=False)
        self.norm = nn.LayerNorm(d_internal)

    def forward(self, e_t: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        """
        e_t : (batch, d_embed)  - token embedding
        h   : (batch, d_state)  - SSM state (read-only, written only by OutputLayer)
        returns z_0 : (batch, d_internal)
        """
        return self.norm(self.proj(torch.cat([e_t, h], dim=-1)))

---
## 2. Depth Gate

### Gate distribution anchoring

The gate MLP maps the current internal representation `z_k` to a scalar score `s_k`.
The routing probability for continuing to the next processing step is:

```
P(go deeper) = Φ(s_k)
```

where Φ is the **standard normal CDF**. The central design requirement is that across a
training batch, the distribution of `s` stays close to N(0,1). This is enforced by the
**KS anchoring loss** (Section 4).

Because the distribution of `s` is anchored:
- Samples near **s = 0** exit or continue with equal probability (50/50 working point)
- Large **negative s** almost always exits (fast inference)
- Large **positive s** almost always continues (deep thinking)

The working point (50% exit) is exactly at the distribution mean *by construction*, with no
additional hyperparameter tuning.

### Why Φ(s) instead of sigmoid(s)?

Graves (2016) ACT uses `sigmoid(s)` with a ponder cost regularizer. The problem: sigmoid(s)
has a stable fixed point near 0 or 1 (the gate saturates), and the ponder cost must be
carefully tuned against the task loss to prevent collapse. By enforcing the shape of the
*entire distribution* via KS divergence, we get:

- No separate regularization coefficient for the gate
- The working point is defined by distribution geometry, not loss balancing
- Inference compute budget can be shifted by a scalar offset μ (Smirnov transform) without retraining:

```
P(go deeper) = Φ(s + μ)    μ > 0 → deeper,  μ < 0 → faster
```

In [ ]:
class GateMLP(nn.Module):
    """
    Depth gate: evaluates whether z_k requires further processing.
    Produces a scalar score s ~ N(0,1) (enforced by KS anchoring loss).
    P(go deeper) = Φ(s)  —  standard normal CDF.

    At inference, apply Smirnov shift μ: P(go deeper) = Φ(s + μ)
    This maps gate scores to a biased distribution without any retraining.
    """
    def __init__(self, d_internal: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_internal),
            nn.Linear(d_internal, d_internal // 2, bias=False),
            nn.GELU(),
            nn.Linear(d_internal // 2, 1, bias=False)
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """z : (batch, d_internal) → s : (batch,) gate scores"""
        return self.net(z).squeeze(-1)

    @staticmethod
    def normal_cdf(s: torch.Tensor) -> torch.Tensor:
        """Standard normal CDF: Φ(s) = 0.5 * (1 + erf(s / √2))"""
        return 0.5 * (1.0 + torch.erf(s / math.sqrt(2.0)))

---
## 3. Shared Processing Layer

The processing layer is **pure compute**: no access to the SSM state, no side effects.
It is applied recurrently: `z_{k+1} = ProcessingLayer(z_k)`, with all recurrent
applications sharing *the same weights*.

### Why shared weights?

Consider two alternatives:

| Architecture | Parameters | Path space |
|---|---|---|
| D distinct layers (standard) | D × P | D paths (one per depth) |
| 1 shared layer applied D times | P | D paths (same weights, different input state) |
| 1 shared MoE, K active, D steps (Phase 2) | P | K^D paths — combinatorial |

A shared MoE layer is active at all depths simultaneously, receiving gradient signal from
all curriculum stages. Routing diversity substitutes for layer diversity, and the
effective computation path space grows far faster than linearly.

In Phase 1, we use a single MLP expert (the monolith). Adding MoE routing in Phase 2
requires no changes to the gate or SSM — only the internals of this layer change.

### Pre-norm residual connection

```
z_{k+1} = z_k + MLP( LayerNorm(z_k) )
```

The residual connection ensures that even if the gate rarely exits early (during curriculum
stage 1), the representation is not destroyed by each pass through the layer. The pre-norm
variant (LayerNorm before the MLP, not after) is more stable in practice.

In [ ]:
class ProcessingLayer(nn.Module):
    """
    Shared compute layer — pure transformation, no SSM state access.
    Applied recurrently: z_{k+1} = z_k + MLP(LayerNorm(z_k))

    All recurrent applications share the same weights (this object is called
    multiple times in the depth loop, never instantiated multiple times).

    Phase 1: single monolithic MLP.
    Phase 2+: replace internals with a Mixture-of-Experts — the interface is identical.
    """
    def __init__(self, d_internal: int, d_mlp: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_internal)
        self.net = nn.Sequential(
            nn.Linear(d_internal, d_mlp, bias=False),
            nn.GELU(),
            nn.Linear(d_mlp, d_internal, bias=False)
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """z : (batch, d_internal) → z_next : (batch, d_internal)"""
        return z + self.net(self.norm(z))

---
## 4. Output Layer and SSM State Update

The output layer performs two distinct operations:

### 4a. Logit projection
```
logits = W_lm · LayerNorm(z_K)
```
During **training** (full unroll), logits are computed at every depth k = 0 … K for the
weighted LM loss. At **inference**, logits are only computed at the actual exit depth.

### 4b. SSM state update
The new SSM state is computed from `z_K` (the fully-processed representation at the
maximum / exit depth). We use a **minimal GRU-style gated update**:

```
g_t  = σ( W_g · [z_K ; h_{t-1}] )          — forget/remember gate
h_t  = g_t ⊙ h_{t-1} + (1 − g_t) ⊙ tanh(W_c · z_K)
```

This allows the model to learn when to carry forward existing state (g → 1) vs replace it
with new content (g → 0), enabling stable gradient flow over long sequences.

### 4c. Soft log-norm regularization

Rather than enforcing hard unitary/near-unitary transitions (as in standard Mamba), we use
a soft penalty in log-norm space applied externally to h_t:

```
L_norm = ( ln ‖h_t‖ )²
```

**Properties:**
- Minimum exactly at unit norm (‖h‖ = 1 → ln‖h‖ = 0)
- Symmetric in log space: 2× oversize and ½× undersize receive equal penalty
- Catches exponential drift: if norm compounds, log grows linearly, penalty escalates early
- The model is **free to deviate** when prediction benefit outweighs the penalty — this is
  the correct trade-off; hard constraints prevent the model from using norm as an information channel

In [ ]:
class OutputLayer(nn.Module):
    """
    Projects internal representation to vocabulary logits and updates the SSM state.
    Executed exactly once per token at the final processing depth z_K.

    get_logits()    — called at every depth during training for weighted LM loss.
    update_state()  — called once per token with z_K to advance h.
    """
    def __init__(self, d_internal: int, d_state: int, vocab_size: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_internal)
        # Weight-tied logit projection (no bias — standard LM practice)
        self.logit_proj = nn.Linear(d_internal, vocab_size, bias=False)

        # Minimal GRU-style gated state update
        self.W_gate      = nn.Linear(d_internal + d_state, d_state, bias=False)
        self.W_candidate = nn.Linear(d_internal, d_state, bias=False)

    def get_logits(self, z: torch.Tensor) -> torch.Tensor:
        """z : (batch, d_internal) → logits : (batch, vocab_size)"""
        return self.logit_proj(self.norm(z))

    def update_state(self, z: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        """
        z : (batch, d_internal)   - final processing depth representation
        h : (batch, d_state)      - previous SSM state
        returns h_new : (batch, d_state)
        """
        gate      = torch.sigmoid(self.W_gate(torch.cat([z, h], dim=-1)))
        candidate = torch.tanh(self.W_candidate(z))
        return gate * h + (1.0 - gate) * candidate

---
## 5. Full Model — AdaptiveSSM

The model wires together the four components above into a single `nn.Module`.

### Per-token forward pass (training — full unroll)

For each token at position t in the sequence:

1. **Embed**: `e_t = Embedding(token_t)`
2. **Input**: `z_0 = InputLayer(e_t, h_{t-1})`
3. **Depth loop** (k = 0 … max_depth − 1):
   - Evaluate gate: `s_k = GateMLP(z_k)`
   - Compute logits: `logits_k = OutputLayer.get_logits(z_k)` (for training loss)
   - Apply processing: `z_{k+1} = ProcessingLayer(z_k)`
4. **Final logits**: `logits_{max_depth} = OutputLayer.get_logits(z_{max_depth})`
5. **State update**: `h_t = OutputLayer.update_state(z_{max_depth}, h_{t-1})`

All gate scores and logits at every depth are returned. The caller (training loop)
assembles the weighted LM loss and the KS anchoring loss from these.

### Phase 2 expansion path
Replacing `ProcessingLayer` with an MoE variant requires **zero changes** to this class.
The gate, SSM backbone, input/output layers remain identical.

In [ ]:
class AdaptiveSSM(nn.Module):
    """
    Adaptive Compute SSM — Phase 1 (monolithic MLP processing layer).

    Architecture (strictly separated roles):
      InputLayer      : (e_t, h_{t-1}) → z_0          [once per token]
      GateMLP         : z_k → scalar s_k               [at each depth]
      ProcessingLayer : z_k → z_{k+1}  (shared weights)[0..D times]
      OutputLayer     : z_K → logits + h_t              [once per token]

    The SSM state is the ONLY inter-token memory.
    Processing layers have NO access to h.
    """
    def __init__(
        self,
        vocab_size  : int,
        d_embed     : int,
        d_state     : int,
        d_internal  : int,
        d_mlp       : int,
        max_depth   : int = 1
    ):
        super().__init__()
        self.d_state   = d_state
        self.max_depth = max_depth

        self.embedding        = nn.Embedding(vocab_size, d_embed)
        self.input_layer      = InputLayer(d_embed, d_state, d_internal)
        self.gate             = GateMLP(d_internal)
        self.processing_layer = ProcessingLayer(d_internal, d_mlp)  # shared — never duplicated
        self.output_layer     = OutputLayer(d_internal, d_state, vocab_size)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def init_state(self, batch_size: int, device) -> torch.Tensor:
        """Zero-initialise the SSM state at the start of a new sequence."""
        return torch.zeros(batch_size, self.d_state, device=device)

    # ------------------------------------------------------------------
    # Training: full-unroll forward for one token position
    # ------------------------------------------------------------------
    def forward_token(
        self,
        token_ids : torch.Tensor,   # (batch,)
        h         : torch.Tensor,   # (batch, d_state)
        max_depth : int
    ):
        """
        Full-unroll forward pass for one token position.
        Always processes to max_depth (training curriculum mode).

        Returns:
            all_logits  : list[(batch, vocab_size)], length = max_depth + 1
            gate_scores : list[(batch,)],            length = max_depth
            h_new       : (batch, d_state)
        """
        e_t = self.embedding(token_ids)          # (batch, d_embed)
        z   = self.input_layer(e_t, h)           # (batch, d_internal) : z_0

        all_logits  = []
        gate_scores = []

        for _ in range(max_depth):
            s_k = self.gate(z)                              # (batch,)
            gate_scores.append(s_k)
            all_logits.append(self.output_layer.get_logits(z))  # logits at depth k
            z = self.processing_layer(z)                   # z_{k+1}

        # Forced exit: logits at final depth (no gate evaluation)
        all_logits.append(self.output_layer.get_logits(z))

        # SSM state update — only from the fully-processed z_K
        h_new = self.output_layer.update_state(z, h)

        return all_logits, gate_scores, h_new

    # ------------------------------------------------------------------
    # Inference: early-exit generation with Smirnov depth control
    # ------------------------------------------------------------------
    @torch.no_grad()
    def generate(
        self,
        prompt_ids     : torch.Tensor,  # (1, prompt_len)
        max_new_tokens : int   = 100,
        mu             : float = 0.0,   # Smirnov shift: >0 = deeper, <0 = faster
        temperature    : float = 1.0,
        top_k          : int   = 50
    ) -> torch.Tensor:
        """
        Autoregressive token generation.
        The shift mu applies the Smirnov (probability integral) transform to gate scores,
        biasing the depth distribution without any model retraining.
        """
        self.eval()
        dev = next(self.parameters()).device
        h = self.init_state(1, dev)
        generated = prompt_ids.to(dev)

        # Warm up SSM state on prompt
        for t in range(generated.shape[1] - 1):
            _, _, h = self.forward_token(generated[:, t], h, self.max_depth)

        for _ in range(max_new_tokens):
            e_t = self.embedding(generated[:, -1])
            z   = self.input_layer(e_t, h)

            # Early-exit depth loop (inference mode)
            for _ in range(self.max_depth):
                s = self.gate(z)
                if torch.rand(1, device=dev).item() > GateMLP.normal_cdf(s + mu).mean().item():
                    break
                z = self.processing_layer(z)

            logits = self.output_layer.get_logits(z) / temperature
            h      = self.output_layer.update_state(z, h)

            if top_k > 0:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1:]] = float('-inf')

            next_tok = torch.multinomial(F.softmax(logits, dim=-1), 1)
            generated = torch.cat([generated, next_tok], dim=1)

        return generated

---
## 6. Training Objective

The total loss combines three terms:

```
L = L_LM  +  λ_KS · D( p_s ‖ N(0,1) )  +  λ_norm · (ln ‖h_t‖)²
```

### 6.1  Weighted LM loss — full unroll, fully differentiable

Every token is always unrolled to the current curriculum max depth.
The cross-entropy loss at each exit depth is weighted by the probability of having exited there:

```
L_LM = Σ_k  P(exit at k) · CE(logits_k, target)

P(exit at 0)         = 1 − Φ(s₀)
P(exit at k)         = Φ(s₀)·…·Φ(s_{k-1}) · (1 − Φ(s_k))    0 < k < D
P(exit at max_depth) = Φ(s₀)·…·Φ(s_{D-1})                    (forced exit)
```

**No REINFORCE or straight-through estimator is needed.** The exit probabilities are products
of Φ(s_k) — each is a smooth, differentiable function of the gate score. Every depth level
receives a gradient on every training step.

### 6.2  KS anchoring loss

We use the **Cramér-von Mises statistic** as a differentiable approximation to the KS distance:

```
L_KS = (1/n) Σᵢ ( F_empirical(sᵢ) − Φ(sᵢ) )²
```

where `sᵢ` are the sorted gate scores from the batch and `F_empirical(sᵢ) = i/n`.
`torch.sort()` is differentiable — gradients pass through to the original gate scores and
back to the gate MLP weights.

The gradient w.r.t. sorted score `sᵢ` is:
```
∂L/∂sᵢ = 2 · ( F_empirical(sᵢ) − Φ(sᵢ) ) · (−φ(sᵢ))
```
This pushes each score toward the N(0,1) quantile corresponding to its rank — the
exact behaviour we want.

### 6.3  Depth curriculum

Training starts with max_depth = 1 (processing layer applied exactly once). The gate begins
learning to distinguish easy from hard tokens only after depth 2 is introduced. This prevents
the gate from collapsing to always-exit before it has seen meaningful processing steps to
compare against.

In [ ]:
def compute_lm_loss(
    all_logits  : list,          # list of (batch, vocab_size), length = max_depth + 1
    gate_scores : list,          # list of (batch,),            length = max_depth
    targets     : torch.Tensor,  # (batch,)
    max_depth   : int
) -> torch.Tensor:
    """
    Full-unroll weighted LM loss.

    L_LM = sum_k  P(exit at k) * CE(logits_k, target)

    Fully differentiable — exit probabilities are products of Phi(s_k),
    each a smooth function of the gate MLP output.
    """
    device = targets.device
    batch  = targets.shape[0]

    p_continue = [GateMLP.normal_cdf(s) for s in gate_scores]  # list of (batch,)

    p_reach    = torch.ones(batch, device=device)   # P(reaching depth k)
    total_loss = torch.zeros(1, device=device)

    for k in range(max_depth):
        p_exit = p_reach * (1.0 - p_continue[k])                         # (batch,)
        ce_k   = F.cross_entropy(all_logits[k], targets, reduction='none')# (batch,)
        total_loss = total_loss + (p_exit * ce_k).mean()
        p_reach    = p_reach * p_continue[k]       # survive to next depth

    # Forced exit — remaining probability mass goes to the deepest level
    ce_final   = F.cross_entropy(all_logits[max_depth], targets, reduction='none')
    total_loss = total_loss + (p_reach * ce_final).mean()

    return total_loss


def ks_anchoring_loss(gate_scores_list: list) -> torch.Tensor:
    scores = torch.cat([s.flatten() for s in gate_scores_list])
    
    mean = scores.mean()
    std  = scores.std()
    
    # Explicit moment penalties - separable and independently tunable
    mean_loss = mean ** 2                  # penalize deviation from 0
    std_loss  = (std - 1.0) ** 2          # penalize deviation from 1
    
    # CvM for higher-order shape (skew, kurtosis)
    sorted_s = torch.sort(scores).values
    n = scores.shape[0]
    empirical_cdf = torch.arange(1, n+1, dtype=scores.dtype, device=scores.device) / n
    target_cdf = 0.5 * (1 + torch.erf(sorted_s / 2**0.5))
    shape_loss = torch.mean((empirical_cdf - target_cdf) ** 2)
    
    return mean_loss, std_loss


def norm_regularization_loss(h: torch.Tensor) -> torch.Tensor:
    """
    Soft log-norm regularization on the SSM state.
    L_norm = mean( ln(||h_t||) )^2

    Minimum at unit norm. Symmetric in log space.
    Gradient = 2*ln||h|| / ||h||  — forceful as norm drifts.
    """
    norms     = torch.norm(h, dim=-1)           # (batch,)
    log_norms = torch.log(norms + 1e-8)
    return torch.mean(log_norms ** 2)

---
## 7. Dataset — TinyStories

[TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) is a synthetic dataset
of short English stories generated by GPT-3.5/4, designed to train small language models.
Each story is typically 100–500 tokens, making it well-suited for the sequential SSM training
loop where we process tokens one-by-one.

We tokenize with the **GPT-2 BPE tokenizer** (vocabulary size 50,257). Each story is
truncated or padded to a fixed `seq_len` and returned as `(input_ids, target_ids)` pairs
for next-token prediction.

### Note on sequence length and memory

The recurrent SSM architecture processes tokens *sequentially*. For TBPTT with `chunk_size`
tokens per backward pass, peak memory scales with `chunk_size × batch_size × model_width`
rather than `seq_len × batch_size × model_width`. This makes long sequences tractable.
Recommended: `seq_len=256`, `chunk_size=32`, `batch_size=8` on a 16 GB GPU.

In [ ]:
class TinyStoriesDataset(Dataset):
    """
    Streams TinyStories from HuggingFace.
    Returns (input_ids, target_ids) tensors of length seq_len for next-token prediction.
    """
    def __init__(
        self,
        split         : str  = 'train',
        seq_len       : int  = 256,
        num_samples   : int  = None,
        tokenizer_name: str  = 'gpt2'
    ):
        self.seq_len   = seq_len
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        print(f'Loading TinyStories [{split}]...')
        self.data = load_dataset('roneneldan/TinyStories', split=split)
        if num_samples is not None:
            self.data = self.data.select(range(min(num_samples, len(self.data))))
        print(f'  {len(self.data):,} stories loaded  |  seq_len={seq_len}')

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        text   = self.data[idx]['text']
        tokens = self.tokenizer(
            text,
            max_length    = self.seq_len + 1,
            padding       = 'max_length',
            truncation    = True,
            return_tensors= 'pt'
        )['input_ids'].squeeze(0)  # (seq_len + 1,)

        return tokens[:-1], tokens[1:]  # input, target

---
## 8. Training Loop — TBPTT

We train with **Truncated Backpropagation Through Time (TBPTT)**.
The sequence is processed in chunks of `chunk_size` tokens. Within each chunk,
the full computation graph is kept and gradients flow through all tokens.
At each chunk boundary, the SSM state `h` is detached — severing the gradient
graph but preserving the state *value* for the next chunk.

This bounds peak memory to `O(chunk_size)` while still allowing the model to learn
temporal dependencies spanning many tokens through the forwarded state.

### Per-chunk optimization step

```
for chunk in sequence:
    zero_grad()
    for t in chunk:
        all_logits, gate_scores, h = model.forward_token(token[t], h, max_depth)
        L_LM   += compute_lm_loss(...)   / chunk_size
        L_norm += norm_reg_loss(h)        / chunk_size
        collect gate_scores
    L_KS = ks_anchoring_loss(gate_scores)
    loss = L_LM + λ_KS·L_KS + λ_norm·L_norm
    loss.backward()
    clip_grad_norm(); optimizer.step()
    h = h.detach()   # ← TBPTT boundary
```

### Metrics logged to Comet ML

| Metric | What to watch for |
|--------|------------------|
| `loss_lm` | Should decrease steadily (target < 3.0 for TinyStories) |
| `loss_ks` | Should stay small (< 0.01 when gate is well-anchored) |
| `loss_norm` | Should stay small (< 0.1); spikes → state norm explosion |
| `gate_mean` | Should stay near 0 |
| `gate_std` | Should stay near 1 |
| `h_norm_mean` | Should stay near 1 |
| `val_ppl` | Perplexity on validation set — primary quality metric |

In [ ]:
def train(config: dict, experiment=None) -> nn.Module:
    """
    Full training loop for AdaptiveSSM.
    TBPTT: backward pass every config['chunk_size'] tokens.
    Mixed precision via torch.cuda.amp.
    """
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    if experiment:
        experiment.log_parameters(config)

    # ── Tokenizer & model ──────────────────────────────────────────────────
    tokenizer  = AutoTokenizer.from_pretrained(config['tokenizer'])
    tokenizer.pad_token = tokenizer.eos_token
    vocab_size = tokenizer.vocab_size

    model = AdaptiveSSM(
        vocab_size  = vocab_size,
        d_embed     = config['d_embed'],
        d_state     = config['d_state'],
        d_internal  = config['d_internal'],
        d_mlp       = config['d_mlp'],
        max_depth   = config['max_depth']
    ).to(dev)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'Model parameters: {n_params:,}')
    if experiment:
        experiment.log_parameter('n_params', n_params)

    # ── Data ───────────────────────────────────────────────────────────────
    train_ds = TinyStoriesDataset('train',      config['seq_len'], config.get('num_train'))
    val_ds   = TinyStoriesDataset('validation', config['seq_len'], config.get('num_val'))

    train_loader = DataLoader(train_ds, batch_size=config['batch_size'],
                              shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'],
                              shuffle=False, num_workers=2, pin_memory=True)

    # ── Optimiser ──────────────────────────────────────────────────────────
    # Muon: 2-D weight matrices of Linear layers only
    # AdamW: Embedding (sparse) + LayerNorm (1-D)
    emb_ids      = {id(p) for p in model.embedding.parameters()}
    muon_params  = [p for p in model.parameters() if p.ndim >= 2 and id(p) not in emb_ids]
    adamw_params = [p for p in model.parameters() if p.ndim < 2 or id(p) in emb_ids]

    optimizer      = torch.optim.Muon(muon_params,  lr=config['lr'])
    optimizer_adamw = torch.optim.AdamW(
        adamw_params, lr=config['lr'],
        weight_decay=config['weight_decay'], betas=(0.9, 0.95)
    )
    steps_per_epoch = len(train_loader) * (config['seq_len'] // config['chunk_size'])
    total_steps     = config['epochs'] * steps_per_epoch
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=config['lr'] * 0.1
    )
    scheduler_adamw = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_adamw, T_max=total_steps, eta_min=config['lr'] * 0.1
    )
    scaler = GradScaler("cuda")   # AMP mixed precision

    global_step = 0

    for epoch in range(config['epochs']):
        model.train()
        epoch_lm = epoch_ks = epoch_norm = n_chunks = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["epochs"]}')

        for input_ids, target_ids in pbar:
            input_ids  = input_ids.to(dev)   # (batch, seq_len)
            target_ids = target_ids.to(dev)
            batch_size = input_ids.shape[0]
            seq_len    = input_ids.shape[1]

            # Fresh SSM state at the start of each story
            h = model.init_state(batch_size, dev)

            # ── TBPTT chunks ───────────────────────────────────────────────
            for chunk_start in range(0, seq_len, config['chunk_size']):
                chunk_end = min(chunk_start + config['chunk_size'], seq_len)

                optimizer.zero_grad()
                optimizer_adamw.zero_grad()
                chunk_lm   = torch.zeros(1, device=dev)
                chunk_norm = torch.zeros(1, device=dev)
                all_gate_scores = []
                n_tok = 0

                with autocast("cuda"):
                    for t in range(chunk_start, chunk_end):
                        all_logits, gate_scores, h_new = model.forward_token(
                            input_ids[:, t], h, config['max_depth']
                        )
                        lm_t   = compute_lm_loss(
                            all_logits, gate_scores, target_ids[:, t], config['max_depth']
                        )
                        norm_t = norm_regularization_loss(h_new)

                        chunk_lm   = chunk_lm   + lm_t
                        chunk_norm = chunk_norm + norm_t
                        all_gate_scores.extend(gate_scores)

                        h     = h_new
                        n_tok += 1

                    chunk_lm   = chunk_lm   / n_tok
                    chunk_norm = chunk_norm / n_tok
                    ks_mean_loss, ks_std_loss    = ks_anchoring_loss(all_gate_scores)
                    ks_loss = ks_mean_loss + ks_std_loss
                    loss = (
                        chunk_lm
                        + config['lambda_ks']   * ks_loss
                        + config['lambda_norm'] * chunk_norm
                    )

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                scaler.unscale_(optimizer_adamw)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
                scaler.step(optimizer)
                scaler.step(optimizer_adamw)
                scaler.update()
                scheduler.step()
                scheduler_adamw.step()

                # Detach SSM state at TBPTT boundary
                h = h.detach()

                epoch_lm   += chunk_lm.item()
                epoch_ks   += ks_loss.item()
                epoch_norm += chunk_norm.item()
                n_chunks   += 1
                global_step += 1

                # ── Logging ──────────────────────────────────────────────
                if experiment and global_step % config['log_every'] == 0:
                    if all_gate_scores:
                        sc = torch.cat([s.detach().flatten() for s in all_gate_scores])
                        experiment.log_metric('gate_mean', sc.mean().item(), step=global_step)
                        experiment.log_metric('gate_std',  sc.std().item(),  step=global_step)

                    h_norm = torch.norm(h, dim=-1).mean().item()
                    experiment.log_metric('h_norm_mean', h_norm,              step=global_step)
                    experiment.log_metric('loss_lm',     chunk_lm.item(),     step=global_step)
                    experiment.log_metric('loss_ks',     ks_loss.item(),      step=global_step)
                    experiment.log_metric('loss_ks_mean', ks_mean_loss.item(), step=global_step)
                    experiment.log_metric('loss_ks_std',  ks_std_loss.item(),  step=global_step)
                    experiment.log_metric('loss_norm',   chunk_norm.item(),   step=global_step)
                    experiment.log_metric('loss_total',  loss.item(),         step=global_step)
                    experiment.log_metric('lr', scheduler.get_last_lr()[0],   step=global_step)

            avg_lm  = epoch_lm  / max(n_chunks, 1)
            avg_ppl = math.exp(min(avg_lm, 20))
            pbar.set_postfix({'lm': f'{avg_lm:.3f}', 'ppl': f'{avg_ppl:.1f}',
                              'ks': f'{epoch_ks/max(n_chunks,1):.4f}'})

        # ── Validation ────────────────────────────────────────────────────
        model.eval()
        val_lm_sum = 0.0
        val_n      = 0

        with torch.no_grad():
            for input_ids, target_ids in tqdm(val_loader, desc='Validation', leave=False):
                input_ids  = input_ids.to(dev)
                target_ids = target_ids.to(dev)
                h = model.init_state(input_ids.shape[0], dev)
                seq_lm = 0.0

                for t in range(input_ids.shape[1]):
                    al, gs, h = model.forward_token(
                        input_ids[:, t], h, config['max_depth']
                    )
                    seq_lm += compute_lm_loss(al, gs, target_ids[:, t],
                                              config['max_depth']).item()

                val_lm_sum += seq_lm / input_ids.shape[1]
                val_n      += 1

        val_lm  = val_lm_sum / max(val_n, 1)
        val_ppl = math.exp(min(val_lm, 20))
        print(f'\nEpoch {epoch+1} | val_loss={val_lm:.4f} | val_ppl={val_ppl:.2f}')

        if experiment:
            experiment.log_metric('val_loss_lm', val_lm,  epoch=epoch)
            experiment.log_metric('val_ppl',     val_ppl, epoch=epoch)

        # ── Checkpoint ───────────────────────────────────────────────────
        ckpt = f'adaptive_ssm_epoch{epoch+1}.pt'
        torch.save({
            'epoch'            : epoch + 1,
            'model_state_dict' : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'optimizer_adamw_state_dict': optimizer_adamw.state_dict(),
            'config'           : config,
            'val_loss'         : val_lm
        }, ckpt)
        print(f'Checkpoint saved: {ckpt}')
        if experiment:
            log_model(experiment, model, f'AdaptiveSSM-epoch{epoch+1}')

    return model

---
## 9. Hyperparameters and Run

The configuration below targets a small prototype suitable for Google Colab's T4 GPU (16 GB).
Increase `num_train` or remove the sample cap once you've confirmed convergence on the subset.

### Depth curriculum reminder
- **Phase 1 (this run)**: `max_depth = 1` — the model always applies one processing step.
  The gate still runs and its distribution is anchored, but there is no adaptive compute yet.
  This validates that the SSM backbone, logit projection, and state update all converge.
- **Phase 2**: set `max_depth = 2` or `3` — the gate now has meaningful choices to learn.

### Key hyperparameter guidance
| Param | Role | Start here |
|-------|------|------------|
| `lambda_ks` | Gate anchoring strength | 0.1 — reduce if gate loss dominates |
| `lambda_norm` | State norm penalty | 0.01 — increase if `h_norm_mean` drifts far from 1 |
| `chunk_size` | TBPTT window | 32 — increase for better temporal credit, watch VRAM |
| `d_state` | SSM memory capacity | 256 for prototype, 512+ for scaling |

In [ ]:
config = {
    # Tokenizer
    'tokenizer'   : 'gpt2',

    # Model dimensions
    'd_embed'     : 128,   # token embedding dimension
    'd_state'     : 256,   # SSM hidden state (sequence memory)
    'd_internal'  : 256,   # internal processing space
    'd_mlp'       : 512,   # processing MLP hidden dimension

    # Depth curriculum — Phase 1: flat at depth 1
    # Increase to 2, then 3 in later phases once this converges
    'max_depth'   : 1,

    # Sequence & batching
    'seq_len'     : 256,   # tokens per story window
    'batch_size'  : 8,     # reduce to 4 if OOM
    'chunk_size'  : 32,    # TBPTT chunk length

    # Optimisation
    'epochs'        : 5,
    'lr'            : 3e-4,
    'weight_decay'  : 0.1,
    'max_grad_norm' : 1.0,

    # Loss weights
    'lambda_ks'   : 0.01,    # KS anchoring strength
    'lambda_norm' : 0.001,   # SSM state log-norm regularization

    # Data subsets (set to None to use the full ~2M-story dataset)
    'num_train'   : 50_000,
    'num_val'     : 2_000,

    # Comet ML logging frequency (in optimizer steps)
    'log_every'   : 50,
}

print('Config:')
for k, v in config.items():
    print(f'  {k:20s} = {v}')

model = train(config, experiment=experiment)

---
## 10. Sample Generation

After training, we can generate text with the model using the `generate()` method.
The `mu` (μ) parameter applies the **Smirnov depth control** at inference:
each gate score `s` is shifted to `s + μ` before sampling the exit decision.

This lets us sweep the thinking depth *continuously* without any retraining:
```
μ = 0.0   →  training-time working point (50% exit at each step)
μ = 2.0   →  ~98% of tokens get the full max_depth processing
μ = -2.0  →  ~98% of tokens exit after zero processing steps
```
(With Phase 1 max_depth=1, the depth variation effect is small — more interesting in Phase 2+.)

In [ ]:
checkpoint = torch.load('adaptive_ssm_epoch1.pt', map_location=device)
config = checkpoint['config']

model = AdaptiveSSM(
    vocab_size  = tokenizer.vocab_size,
    d_embed     = config['d_embed'],
    d_state     = config['d_state'],
    d_internal  = config['d_internal'],
    d_mlp       = config['d_mlp'],
    max_depth   = config['max_depth']
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("Model loaded successfully from checkpoint.")

In [ ]:
def generate_sample(model, prompt: str, tokenizer, max_new_tokens=150,
                    mu=0.0, temperature=0.9, top_k=50):
    dev    = next(model.parameters()).device
    tokens = tokenizer.encode(prompt, return_tensors='pt')
    output = model.generate(tokens, max_new_tokens=max_new_tokens,
                            mu=mu, temperature=temperature, top_k=top_k)
    return tokenizer.decode(output[0], skip_special_tokens=True)


tokenizer = AutoTokenizer.from_pretrained(config['tokenizer'])
tokenizer.pad_token = tokenizer.eos_token

prompt = 'Once upon a time, a little girl found a golden key in the forest.'

print('=' * 60)
print(f'Prompt: {prompt}')
print('=' * 60)

for mu, label in [(-1.0, 'Fast (mu=-1)'), (0.0, 'Default (mu=0)'), (1.0, 'Deep (mu=+1)')]:
    print(f'\n--- {label} ---')
    print(generate_sample(model, prompt, tokenizer, mu=mu))